# DDSP Model Evaluation

Evaluate a trained DDSP autoencoder. **Just set `CONFIG_NAME`** below — the model,
dataset, sample-rate, channel count and resampling factor are all resolved from
`configs/<CONFIG_NAME>.yaml` and the matching checkpoint via `eval_utils`.

Reconstruction is measured with:
- **MSE** and **multi-resolution STFT loss** (auraloss),
- overlaid magnitude spectra (original vs. resynthesised),
- waveform comparison and stereo playback.

In [ ]:
CONFIG_NAME      = "topiary_stereo"   # configs/<CONFIG_NAME>.yaml
CHUNK_DURATION_S = 8.0                 # eval chunk length (None -> use config value)
NUM_SAMPLES      = 3
DEVICE           = "cuda"             # "cuda" or "cpu"

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repo root, so `import eval_utils` and `ddsp` work

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import auraloss.freq
from IPython.display import Audio, display

import eval_utils as E

sns.set_theme(style="darkgrid", context="notebook", palette="muted")
torch.set_grad_enabled(False)
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")

## 1 - Load config, model and dataset (config name is all you need)

In [ ]:
cfg  = E.load_config(CONFIG_NAME)
ddsp = E.load_ddsp(cfg, device=DEVICE)

fs         = int(cfg["audio"]["fs"])
n_channels = int(ddsp.n_channels)
print(f"fs={fs}  n_channels={n_channels}  resampling_factor={ddsp.resampling_factor}")
print(f"feature_dim={ddsp.feature_dim}  latent_size={ddsp.latent_size}")
print(f"synths: {[s.jit_name for s in ddsp.synths]}  |  total synth params: {ddsp._total_synth_params}")

dataset = E.load_dataset(cfg, ddsp, chunk_duration_s=CHUNK_DURATION_S, device=DEVICE)
print(f"Dataset: {len(dataset)} chunks of {CHUNK_DURATION_S or cfg['audio']['chunk_duration_s']}s")

## 2 - Training stats (final logged losses)

Pulled from the synth's TensorBoard run. `val_loss` is the monitored objective;
`MultiResolutionSTFTLoss` is the perceptual spectral term, `kld` the latent KL,
`adv_*` the adversarial terms (0 until their start epochs).

In [ ]:
def show(stats, keys, width=32):
    for k in keys:
        if k in stats:
            s = stats[k]
            print(f"  {k:{width}s} last={s['last']:.5f}  (min={s['min']:.5f}  max={s['max']:.5f}  n={s['n']})")

try:
    ss = E.synth_stats(cfg)
    print("DDSP SYNTH")
    show(ss, ["val_loss", "val/MultiResolutionSTFTLoss", "ddsp_loss", "recons_loss",
              "kld", "beta", "adv_g", "adv_d", "epoch"])
except Exception as e:
    print("synth stats unavailable:", e)

## 3 - Forward pass and reconstruction metrics

In [ ]:
fft_sizes = np.array([2053, 1021, 509, 257, 129, 65, 33])
mrstft = auraloss.freq.MultiResolutionSTFTLoss(
    fft_sizes=fft_sizes.tolist(),
    hop_sizes=(fft_sizes // 4).tolist(),
    win_lengths=fft_sizes.tolist(),
    perceptual_weighting=True,
    sample_rate=fs,
).to(DEVICE)


def evaluate_sample(idx: int):
    audio, features = dataset[idx]                 # audio [n_ch, T], features [T_ctl, D]
    y = E.autoencode(ddsp, audio, features)         # [1, n_ch, T]
    x = audio.unsqueeze(0).to(y.device)             # [1, n_ch, T]

    T = min(x.shape[-1], y.shape[-1])
    x, y = x[..., :T], y[..., :T]

    mse = torch.nn.functional.mse_loss(y, x).item()
    stft = mrstft(y, x).item()
    return {"idx": idx, "mse": mse, "mrstft": stft,
            "x": x[0].cpu().numpy(), "y": y[0].cpu().numpy()}   # [n_ch, T]


rng = np.random.default_rng(42)
indices = rng.choice(len(dataset), size=min(NUM_SAMPLES, len(dataset)), replace=False)
results = [evaluate_sample(int(i)) for i in indices]
for r in results:
    print(f"Sample {r['idx']:>4d}  |  MSE = {r['mse']:.6f}  |  MRSTFT = {r['mrstft']:.4f}")
print("=" * 56)
print(f"Average       |  MSE = {np.mean([r['mse'] for r in results]):.6f}  "
      f"|  MRSTFT = {np.mean([r['mrstft'] for r in results]):.4f}")

## 4 - Magnitude spectrum (mono downmix: original vs. resynthesised)

In [ ]:
def plot_spectrum(x, y, sr, title=""):
    x, y = np.mean(x, axis=0), np.mean(y, axis=0)   # downmix [n_ch, T] -> [T]
    n_fft = 4096
    freqs = np.fft.rfftfreq(n_fft, d=1.0 / sr)
    X = 20 * np.log10(np.abs(np.fft.rfft(x, n=n_fft)) + 1e-8)
    Y = 20 * np.log10(np.abs(np.fft.rfft(y, n=n_fft)) + 1e-8)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(freqs, X, color=sns.color_palette()[0], alpha=0.8, lw=0.8, label="Original")
    ax.plot(freqs, Y, color=sns.color_palette()[3], alpha=0.8, lw=0.8, label="Resynth")
    ax.set(xlabel="Frequency (Hz)", ylabel="Magnitude (dB)", xlim=(0, sr / 2),
           title=title or "Magnitude spectrum")
    ax.legend(frameon=True); sns.despine(fig=fig); plt.tight_layout(); plt.show()


for r in results:
    plot_spectrum(r["x"], r["y"], fs,
                  title=f"Sample {r['idx']}  |  MSE={r['mse']:.6f}  MRSTFT={r['mrstft']:.4f}")

## 5 - Waveform comparison (channel 0)

In [ ]:
def plot_waveforms(x, y, sr, title=""):
    x0, y0 = x[0], y[0]                              # channel 0
    t = np.arange(len(x0)) / sr
    fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True, gridspec_kw={"hspace": 0.25})
    ax[0].plot(t, x0, color=sns.color_palette()[0], lw=0.4); ax[0].set(title="Original", ylabel="Amp")
    ax[1].plot(t, y0, color=sns.color_palette()[3], lw=0.4)
    ax[1].set(title="Resynth", ylabel="Amp", xlabel="Time (s)")
    lim = max(np.abs(x0).max(), np.abs(y0).max()) * 1.1
    for a in ax: a.set_ylim(-lim, lim)
    fig.suptitle(title, y=1.02); sns.despine(fig=fig); plt.tight_layout(); plt.show()


for r in results:
    plot_waveforms(r["x"], r["y"], fs,
                   title=f"Sample {r['idx']}  |  MSE={r['mse']:.6f}  MRSTFT={r['mrstft']:.4f}")

## 6 - Audio playback (stereo where applicable)

In [ ]:
for r in results:
    print(f"\n=== Sample {r['idx']}  (MSE={r['mse']:.6f}  MRSTFT={r['mrstft']:.4f}) ===")
    print("Original:");  display(Audio(r["x"], rate=fs))
    print("Resynth:");   display(Audio(r["y"], rate=fs))